# Refinamiento de la CRNN pequeña con LightGBM DART

**Resultado sobre Telemetry y Cassette.** En las cinco rondas, la mezcla 75/25 obtuvo 80,86 % de F1 macro con la red original, 80,87 % con la residual y 80,83 % con la multiescala. La multiescala casi duplicó los parámetros de la original y tardó 15,3–17,6 segundos por ronda con las características ya preparadas. Se mantiene la CRNN original como referencia.

Se comparan la CRNN original y dos variantes. La residual ensancha la GRU y usa 20 épocas. La nueva CRNN multiescala procesa en paralelo patrones de 3 y de 7 segmentos antes de la GRU. Mantiene 12 épocas y la GRU de 32 unidades por dirección del control. La hipótesis es captar patrones cortos y largos sin asumir el coste de una red sobre señal cruda. La configuración y el reparto 75/25 se fijan antes de puntuar la variante.

En cada segmento los dos clasificadores emiten cinco probabilidades en el mismo orden de etapas. Para cada etapa se suma 0,75 veces la probabilidad del DART y 0,25 veces la de la CRNN. La predicción es la etapa con mayor suma. El F1 global reúne ambas cohortes y las cinco rondas de prueba por persona.


In [1]:
import json
import sys
from pathlib import Path
from tempfile import gettempdir
import numpy as np
import pandas as pd
from IPython.display import display
from lightgbm import Booster
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix, f1_score

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'data.dvc').is_file())
sys.path.insert(0, str(ROOT / 'packages/sleep-staging/src'))
sys.path.insert(0, str(ROOT))
from sleep_staging import PreprocessingConfig, PreprocessingPipeline
from sleep_staging.datasets import discover_sleep_telemetry_records, discover_sleep_cassette_records
from sleep_staging.training import load_or_build_supervised_dataset


ref = ROOT / 'data/experimentos/lightgbm_st_sc'
spec = json.loads((ref / 'experimento.json').read_text('utf-8'))
clases = np.array(spec['classes'])

def datos(registros, cohorte, cache):
    cfg = spec['preprocessing_telemetry' if cohorte == 'Telemetry' else 'preprocessing_cassette']
    ds = load_or_build_supervised_dataset(registros, PreprocessingPipeline(PreprocessingConfig(**cfg)), cache, verbose=False)
    x = ds.features.astype('float32').reindex(columns=spec['feature_columns'])
    m = ds.metadata[['record_id','subject_id','epoch_index']].copy()
    m['cohorte'], m['real'] = cohorte, ds.labels.to_numpy()
    return x, m
st = discover_sleep_telemetry_records(ROOT/'data/sleep-telemetry')
sc = discover_sleep_cassette_records(ROOT/'data/sleep-cassette')
partes = [datos(st,'Telemetry',ROOT/'data/processed/supervised')]
partes.extend(datos([r],'Cassette',Path(gettempdir())/'sleep-edfx-sc-features-v1') for r in sc)
X = pd.concat([x for x,_ in partes], ignore_index=True)
meta = pd.concat([m for _,m in partes], ignore_index=True)
del partes
assert not meta.duplicated(['record_id','epoch_index']).any() and set(meta.real)==set(clases)

C:\Users\sebas\AppData\Local\Temp\ipykernel_25532\803375649.py:34: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  X = pd.concat([x for x,_ in partes], ignore_index=True)


## Redes sobre las mismas características y ventanas

In [2]:
import tensorflow as tf
from sklearn.preprocessing import RobustScaler
from time import perf_counter

try:
    tf.config.set_visible_devices([], 'GPU')
    tf.config.threading.set_intra_op_parallelism_threads(4)
    tf.config.threading.set_inter_op_parallelism_threads(2)
except RuntimeError:
    pass  # Un kernel que ya inicializó TensorFlow conserva su configuración.
instantaneas = [c for c in X if c.endswith('_t0')]
base_neural = X[instantaneas].to_numpy(dtype='float32')
codigos = pd.Categorical(meta.real, categories=clases).codes.astype('int32')

def ventanas(datos):
    indices = []
    for _, noche in datos.groupby('record_id'):
        noche = noche.sort_values('epoch_index')
        for tramo in np.split(noche.index.to_numpy(), np.flatnonzero(np.diff(noche.epoch_index)!=1)+1):
            for inicio in range(0, len(tramo), 32):
                v = tramo[inicio:inicio+64]
                indices.append(np.pad(v,(0,64-len(v)),constant_values=-1))
                if inicio+64 >= len(tramo):
                    break
    return np.stack(indices)

def red_secuencial(dim):
    entrada = tf.keras.Input((64,dim))
    # La última variable indica qué posiciones contienen un segmento real.
    mascara = tf.keras.ops.greater(entrada[:,:,-1],0)
    z = tf.keras.layers.Conv1D(48,3,padding='same',activation='relu')(entrada)
    z = tf.keras.layers.Dropout(0.15)(z)
    z = tf.keras.layers.Bidirectional(tf.keras.layers.GRU(32,return_sequences=True))(z, mask=mascara)
    z = tf.keras.layers.Dropout(0.2)(z)
    salida = tf.keras.layers.Dense(len(clases),activation='softmax')(z)
    red = tf.keras.Model(entrada,salida)
    red.compile(optimizer=tf.keras.optimizers.Adam(0.001),loss='sparse_categorical_crossentropy')
    return red


def red_residual(dim):
    entrada = tf.keras.Input((64,dim))
    mascara = tf.keras.ops.greater(entrada[:,:,-1],0)
    z = tf.keras.layers.Conv1D(48,3,padding='same',activation='relu')(entrada)
    detalle = tf.keras.layers.Conv1D(48,3,dilation_rate=3,padding='same',activation='relu')(z)
    z = tf.keras.layers.Add()([z,detalle])
    z = tf.keras.layers.LayerNormalization()(z)
    z = tf.keras.layers.Dropout(0.15)(z)
    z = tf.keras.layers.Bidirectional(tf.keras.layers.GRU(48,return_sequences=True))(z,mask=mascara)
    z = tf.keras.layers.Dropout(0.2)(z)
    salida = tf.keras.layers.Dense(len(clases),activation='softmax')(z)
    red = tf.keras.Model(entrada,salida)
    red.compile(optimizer=tf.keras.optimizers.Adam(0.001),loss='sparse_categorical_crossentropy')
    return red

def red_multiescala(dim):
    entrada = tf.keras.Input((64,dim))
    mascara = tf.keras.ops.greater(entrada[:,:,-1],0)
    corta = tf.keras.layers.Conv1D(32,3,padding='same',activation='relu')(entrada)
    larga = tf.keras.layers.Conv1D(32,7,padding='same',activation='relu')(entrada)
    z = tf.keras.layers.Concatenate()([corta,larga])
    z = tf.keras.layers.Dropout(0.15)(z)
    z = tf.keras.layers.Bidirectional(tf.keras.layers.GRU(32,return_sequences=True))(z,mask=mascara)
    z = tf.keras.layers.Dropout(0.2)(z)
    salida = tf.keras.layers.Dense(len(clases),activation='softmax')(z)
    red = tf.keras.Model(entrada,salida)
    red.compile(optimizer=tf.keras.optimizers.Adam(0.001),loss='sparse_categorical_crossentropy')
    return red


## Validación por persona

Cada ronda reserva todas las noches de las personas de prueba. El escalador, las ponderaciones de clase y los ajustes neuronales usan solo las personas de entrenamiento. Se reconstruyen las mismas particiones estratificadas por cohorte del experimento original. Se reutiliza el DART de cada ronda cuando está disponible. Los vectores de probabilidad de los tres modelos usan el mismo orden de clases.

In [3]:
import hashlib
import os
from tempfile import NamedTemporaryFile
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold

archivo_nb = ROOT/'Notebooks/experiments/lightgbm_transiciones_sueno.ipynb'
codigo = [c['source'] for c in json.loads(archivo_nb.read_text('utf-8'))['cells'] if c['cell_type']=='code']
firma = hashlib.sha256(json.dumps([codigo,lgb.__version__,tf.__version__]).encode())
firma.update((ROOT/'data.dvc').read_bytes())
for fuente in [ROOT/'scripts/compare_cassette_hybrid.py', *sorted((ROOT/'packages/sleep-staging/src/sleep_staging').rglob('*.py'))]:
    firma.update(fuente.read_bytes())
# La caché depende del código de lightgbm_transiciones_sueno, los datos y las versiones.
aliases = json.loads(archivo_nb.read_text('utf-8'))['metadata']['cache_tras_traslado']
cache_cv = ROOT/'.sleep-cv/fusion-crnn'/aliases.get(firma.hexdigest(), firma.hexdigest()[:16])
cache_cv.mkdir(parents=True,exist_ok=True)
personas = meta[['subject_id','cohorte']].drop_duplicates().sort_values(['cohorte','subject_id'])
assert personas.subject_id.is_unique
rondas = [set(personas.iloc[b].subject_id) for _,b in StratifiedKFold(5,shuffle=True,random_state=42).split(personas,personas.cohorte)]
assert len(set.union(*rondas)) == sum(map(len,rondas)) == meta.subject_id.nunique()


original_local = ROOT / '.sleep-cv/8f83a22b1d3a65d3'
filas_oof = []
ejemplos = []
tiempos = []
def entrenar_probar(numero):
    es_test = meta.subject_id.isin(rondas[numero-1])
    tr, te = meta.loc[~es_test], meta.loc[es_test]
    assert set(tr.subject_id).isdisjoint(te.subject_id)
    ruta = cache_cv/f'dart-{numero}.txt'
    if numero==1 and not ruta.exists() and (original_local/'tiempos-ronda-1/dart.txt').exists():
        ruta = original_local/'tiempos-ronda-1/dart.txt'
        assert json.loads((ruta.parent/'mediciones.json').read_text('utf-8'))['clases_dart']==clases.tolist()
    if not ruta.exists():
        ajuste = lgb.LGBMClassifier(**dict(spec['parameters'],n_jobs=6)).fit(X.loc[tr.index],tr.real)
        assert np.array_equal(ajuste.classes_,clases)
        with NamedTemporaryFile(dir=cache_cv,delete=False) as tmp:
            temporal=Path(tmp.name)
        ajuste.booster_.save_model(str(temporal))
        os.replace(temporal,ruta)
    tabular = Booster(model_file=str(ruta)).predict(X.loc[te.index],num_threads=4)
    assert np.allclose(tabular.sum(1),1)
    m=te.copy()
    m['ronda']=numero
    m['DART']=clases[tabular.argmax(1)]
    escala = RobustScaler(quantile_range=(10,90)).fit(base_neural[~es_test])
    z = np.nan_to_num(np.clip(escala.transform(base_neural),-5,5)).astype('float32')
    z = np.column_stack([z,meta.cohorte.eq('Cassette').to_numpy(dtype='float32'),np.ones(len(z),dtype='float32')])
    entradas = np.vstack([z,np.zeros((1,z.shape[1]),dtype='float32')])
    a,b = ventanas(tr),ventanas(te)
    assert set(a[a>=0])==set(tr.index) and set(b[b>=0])==set(te.index)
    counts=np.bincount(codigos[tr.index],minlength=len(clases))
    w=(counts.mean()/counts)**0.35
    w/=np.average(w,weights=counts)
    reales=codigos[np.maximum(a,0)]
    train_x, test_x = entradas[a], entradas[b]
    for nombre,construir,epocas in [('Original',red_secuencial,12),('Residual',red_residual,20),('Multiescala',red_multiescala,12)]:
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(42)
        red=construir(z.shape[1])
        callbacks=[]
        if nombre=='Residual':
            callbacks=[tf.keras.callbacks.LearningRateScheduler(lambda e: 0.001 if e<12 else 0.00035)]
        inicio=perf_counter()
        red.fit(train_x,reales,sample_weight=(w[reales]*(a>=0)).astype('float32'),
                epochs=epocas,batch_size=32,verbose=0,callbacks=callbacks)
        tiempos.append([numero,nombre,perf_counter()-inicio,red.count_params()])
        pred=red.predict(test_x,batch_size=64,verbose=0)
        suma=np.zeros((len(meta),len(clases)))
        votos=np.zeros(len(meta))
        valido=b>=0
        np.add.at(suma,b[valido],pred[valido])
        np.add.at(votos,b[valido],1)
        neural=suma[te.index]/votos[te.index,None]
        mezcla=.75*tabular+.25*neural
        assert np.isfinite(mezcla).all() and np.allclose(mezcla.sum(1),1)
        m[nombre]=clases[neural.argmax(1)]
        m['DART + '+nombre]=clases[mezcla.argmax(1)]
        if numero==1 and nombre=='Original':
            cambios=np.flatnonzero(tabular.argmax(1)!=mezcla.argmax(1))
            j=int(cambios[0]) if len(cambios) else 0
            ejemplos.append((te.iloc[j].record_id,int(te.iloc[j].epoch_index),
                 pd.DataFrame({'DART':tabular[j],'CRNN':neural[j],'75/25':mezcla[j]},index=clases)))
    anterior=cache_cv/f'ronda-{numero}.parquet'
    if anterior.exists():
        claves=['subject_id','record_id','epoch_index','real']
        pares=m.merge(pd.read_parquet(anterior)[claves+['DART','DART + CRNN']],on=claves,validate='one_to_one')
        assert len(pares)==len(m) and np.array_equal(pares.DART_x,pares.DART_y)
        print('Coincidencia con el control 75/25 guardado:',np.mean(pares['DART + Original']==pares['DART + CRNN']))
    filas_oof.append(m)
    print(f'Ronda {numero} · {len(rondas[numero-1])} personas · {len(m)} segmentos')

In [4]:
entrenar_probar(1)

Coincidencia con el control 75/25 guardado: 1.0
Ronda 1 · 7 personas · 15893 segmentos


In [5]:
entrenar_probar(2)

Coincidencia con el control 75/25 guardado: 1.0
Ronda 2 · 7 personas · 15763 segmentos


In [6]:
entrenar_probar(3)

Coincidencia con el control 75/25 guardado: 1.0
Ronda 3 · 7 personas · 15409 segmentos


In [7]:
entrenar_probar(4)

Coincidencia con el control 75/25 guardado: 1.0
Ronda 4 · 7 personas · 14989 segmentos


In [8]:
entrenar_probar(5)

Coincidencia con el control 75/25 guardado: 1.0
Ronda 5 · 6 personas · 13558 segmentos


## Resultados sobre segmentos fuera de muestra

El F1 global se calcula reuniendo todas las predicciones, calculando un F1 para cada una de las cinco etapas y promediando esos cinco valores. Se informan también las dos cohortes por separado, las rondas y un intervalo orientativo para la diferencia entre mezclas, mediante 5.000 remuestreos pareados de personas dentro de cada cohorte.

In [9]:
oof=pd.concat(filas_oof,ignore_index=True)
assert len(oof)==len(meta) and oof.groupby('subject_id').ronda.nunique().eq(1).all()
assert not oof.duplicated(['record_id','epoch_index']).any()
def puntuar(m,modelo):
    return f1_score(m.real,m[modelo],labels=clases,average='macro',zero_division=0)
modelos=['DART','Original','Residual','Multiescala','DART + Residual','DART + Original','DART + Multiescala']
rows=[]
for grupo,m in [('Ambas',oof),*oof.groupby('cohorte')]:
    for modelo in modelos:
        rows.append([grupo,modelo,m.subject_id.nunique(),m.record_id.nunique(),len(m),puntuar(m,modelo),
                     accuracy_score(m.real,m[modelo]),f1_score(m.real,m[modelo],labels=['N1'],average=None)[0]])
display(pd.DataFrame(rows,columns=['Cohorte','Modelo','Personas','Noches','Segmentos','F1 macro','Exactitud','F1 N1']).round(4))
display(pd.DataFrame([[i,*[puntuar(m,k) for k in modelos]] for i,m in oof.groupby('ronda')],columns=['Ronda',*modelos]).round(4))
rng=np.random.default_rng(42)
remuestras=[]
for _,g in oof.groupby('cohorte'):
    cms=np.array([[confusion_matrix(s.real,s[k],labels=clases) for k in modelos[-2:]] for _,s in g.groupby('subject_id')])
    remuestras.append(cms[rng.integers(len(cms),size=(5000,len(cms)))].sum(axis=1))
cm=sum(remuestras)
den=cm.sum(axis=-1)+cm.sum(axis=-2)
scores=np.divide(2*np.diagonal(cm,axis1=-2,axis2=-1),den,out=np.zeros(den.shape),where=den!=0).mean(axis=-1)
print('Diferencia de F1 entre la mezcla multiescala y la original:',round(puntuar(oof,modelos[-1])-puntuar(oof,modelos[-2]),5))
print('IC 95 % de la diferencia:',np.quantile(scores[:,1]-scores[:,0],[.025,.975]).round(5))
display(pd.DataFrame(tiempos,columns=['Ronda','CRNN','Ajuste (s)','Parámetros']).round(1))
rid,epoch,tabla=ejemplos[0]
print(f'Ejemplo de combinación: registro {rid}, segmento {epoch} (índice desde cero).')
display(tabla.round(4))

,Cohorte,Modelo,Personas,Noches,Segmentos,F1 macro,Exactitud,F1 N1
0,Ambas,DART,34,56,75612,0.8010,0.8596,0.5425
1,Ambas,Original,34,56,75612,0.7956,0.8590,0.5319
2,Ambas,Residual,34,56,75612,0.7939,0.8585,0.5201
3,Ambas,Multiescala,34,56,75612,0.7956,0.8603,0.5268
4,Ambas,DART + Residual,34,56,75612,0.8087,0.8676,0.5544
5,Ambas,DART + Original,34,56,75612,0.8086,0.8669,0.5563
6,Ambas,DART + Multiescala,34,56,75612,0.8083,0.8668,0.5557
7,Cassette,DART,12,12,32921,0.7768,0.9144,0.4409
8,Cassette,Original,12,12,32921,0.7733,0.9162,0.4407
9,Cassette,Residual,12,12,32921,0.7607,0.9118,0.4086


,Ronda,DART,Original,Residual,Multiescala,DART + Residual,DART + Original,DART + Multiescala
0,1,0.8194,0.8052,0.8092,0.8186,0.8297,0.8274,0.8297
1,2,0.8025,0.8054,0.7976,0.8058,0.8124,0.8155,0.8132
2,3,0.8075,0.8076,0.8027,0.8042,0.8181,0.8181,0.8173
3,4,0.8071,0.8053,0.7993,0.7938,0.8117,0.8128,0.8098
4,5,0.7598,0.7402,0.7483,0.7394,0.7621,0.7599,0.7617


Diferencia de F1 entre la mezcla multiescala y la original: -0.00037
IC 95 % de la diferencia: [-0.00217  0.0015 ]


,Ronda,CRNN,Ajuste (s),Parámetros
0,1,Original,14.1,53125
1,1,Residual,26.1,72821
2,1,Multiescala,15.3,101445
3,2,Original,14.0,53125
4,2,Residual,25.9,72821
5,2,Multiescala,15.4,101445
6,3,Original,14.1,53125
7,3,Residual,25.9,72821
8,3,Multiescala,15.3,101445
9,4,Original,14.3,53125


Ejemplo de combinación: registro ST7051, segmento 56 (índice desde cero).


,DART,CRNN,75/25
N1,0.5453,0.2767,0.4781
N2,0.4211,0.7189,0.4956
N3,0.0062,0.0008,0.0049
REM,0.0083,0.0031,0.0070
W,0.0191,0.0005,0.0144
